In [1]:
# A2C scaffold for LunarLander-v3.
# Fill in the A2C loss + training loop below, then rerun from the top.

from dataclasses import dataclass
from pathlib import Path

import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch

In [2]:
@dataclass
class A2CConfig:
    env_id: str = "LunarLander-v3"
    seed: int = 42
    learning_rate: float = 3e-4
    gamma: float = 0.99
    value_loss_coef: float = 0.5
    entropy_coef: float = 0.01
    max_grad_norm: float = 0.5
    train_episodes: int = 1_000
    max_steps_per_episode: int = 1_000
    eval_episodes: int = 5
    video_path: str = "lunar_lander_a2c.mp4"


def make_env(env_id: str, seed: int | None = None, render_mode: str | None = None):
    env = gym.make(env_id, render_mode=render_mode)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)
    return env


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = A2CConfig()

train_env = make_env(config.env_id, seed=config.seed)
eval_env = make_env(config.env_id, seed=config.seed + 1, render_mode="rgb_array")

obs_dim = train_env.observation_space.shape[0]
action_dim = train_env.action_space.n

print(f"Using device: {device}")
print(f"Observation dim: {obs_dim}, actions: {action_dim}")

/home/ubuntu/repos/deep-rl/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [ ]:
from model import Policy

torch.manual_seed(config.seed)
np.random.seed(config.seed)

model = Policy(in_dim=obs_dim, out_dim=action_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

model

np.int64(4)

In [ ]:
def to_tensor(obs: np.ndarray) -> torch.Tensor:
    return torch.as_tensor(obs, dtype=torch.float32, device=device)


@torch.no_grad()
def evaluate_policy(env, policy, episodes=5, deterministic=True):
    episode_returns = []

    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            action, _, _, _ = policy.act(to_tensor(obs), deterministic=deterministic)
            obs, reward, terminated, truncated, _ = env.step(int(action.item()))
            total_reward += reward
            done = terminated or truncated

        episode_returns.append(total_reward)

    return {
        "mean_return": float(np.mean(episode_returns)),
        "std_return": float(np.std(episode_returns)),
        "episode_returns": episode_returns,
    }


def plot_rewards(rewards, window=50):
    if not rewards:
        print("No rewards to plot yet.")
        return

    plt.figure(figsize=(10, 4))
    plt.plot(rewards, label="episode return", alpha=0.4)

    if len(rewards) >= window:
        running = np.convolve(rewards, np.ones(window) / window, mode="valid")
        plt.plot(range(window - 1, len(rewards)), running, label=f"{window}-episode mean")

    plt.title("Training rewards")
    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.legend()
    plt.show()

In [ ]:
def compute_a2c_loss(
    rewards,
    values,
    log_probs,
    entropies,
    dones,
    gamma,
    value_loss_coef,
    entropy_coef,
):
    """
    TODO:
    1. Compute discounted returns or bootstrap targets.
    2. Compute advantages = returns - values.
    3. Build actor loss, critic loss, and entropy bonus.
    4. Return total_loss and a metrics dict.
    """
    raise NotImplementedError("Implement the A2C loss here.")

In [ ]:
from torch.distributions import Categorical
import numpy as np

episodes = 10000
lr = 1e-5
episode_rewards = [] # we want to calculate the reward per episode
optim = torch.optim.AdamW(model.parameters(), lr=lr)

for eps in range(episodes):
    state, info = env.reset()
    done = False
    step = 0
    cur_eps_reward = 0 

    while done is False:
        logits = model(torch.from_numpy(state))
        distribution = Categorical(logits=logits)
        # action_probs = torch.softmax(logits, dim=-1)
        action = distribution.sample()
        log_prob = distribution.log_prob(action)

        state, reward, terminated, truncated, info = env.step(action)
        cur_eps_reward += reward
        done = truncated or terminated
    episode_rewards.append(cur_eps_reward)

mean_rewards = np.mean(episode_rewards)
std_rewards = np.std(episode_rewards)


tensor([0])
tensor([0])
tensor([0])
tensor([3])
tensor([0])
tensor([2])
tensor([0])
tensor([3])
tensor([1])
tensor([3])
tensor([2])
tensor([1])
tensor([2])
tensor([2])
tensor([3])
tensor([2])
tensor([3])
tensor([2])
tensor([0])
tensor([0])
tensor([0])
tensor([1])
tensor([2])
tensor([1])
tensor([3])
tensor([3])
tensor([3])
tensor([2])
tensor([3])
tensor([3])
tensor([3])
tensor([3])
tensor([2])
tensor([2])
tensor([0])
tensor([1])
tensor([3])
tensor([3])
tensor([2])
tensor([3])
tensor([1])
tensor([1])
tensor([2])
tensor([3])
tensor([3])
tensor([3])
tensor([0])
tensor([0])
tensor([3])
tensor([0])
tensor([1])
tensor([3])
tensor([0])
tensor([3])
tensor([0])
tensor([3])
tensor([1])
tensor([1])
tensor([2])
tensor([3])
tensor([0])
tensor([3])
tensor([2])
tensor([1])
tensor([0])
tensor([0])
tensor([1])
tensor([2])
tensor([0])
tensor([2])
tensor([2])
tensor([0])
tensor([3])
tensor([0])
tensor([3])
tensor([2])
tensor([0])
tensor([2])
tensor([3])
tensor([2])
tensor([2])
tensor([3])
tensor([2])
tens

In [4]:
@torch.no_grad()
def record_video(env_id, policy, output_path, seed=0, fps=30):
    env = make_env(env_id, seed=seed, render_mode="rgb_array")
    frames = []

    obs, _ = env.reset()
    done = False
    while not done:
        frames.append(env.render())
        action, _, _, _ = policy.act(to_tensor(obs), deterministic=True)
        obs, _, terminated, truncated, _ = env.step(int(action.item()))
        done = terminated or truncated

    frames.append(env.render())
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(output_path, frames, fps=fps)
    env.close()
    return output_path

[-0.00316105  1.4105023  -0.15988626 -0.02211873  0.0036302   0.0358447
  0.          0.        ]
-0.25635202221894815


In [ ]:
# Example workflow after you finish the placeholders:
#
# history = train_a2c(train_env, model, optimizer, config)
# plot_rewards(history["episode_returns"])
#
# eval_metrics = evaluate_policy(eval_env, model, episodes=config.eval_episodes)
# print(eval_metrics)
#
# video_path = record_video(config.env_id, model, config.video_path, seed=config.seed + 123)
# print(f"Saved video to {video_path}")
#
# Rerun this notebook from the top after filling in the A2C cells.